# Q(V, φ) parametrization

Collected charge **Q** of ETL LGADs vs bias **V** and fluence **φ**. Data is read directly from `parametrization_data.xlsx` (one sheet per production).

Modules: `config.py` (settings) · `data_io.py` (Excel reader) · `model.py` (fits) · `plots.py` (ROOT plots).

**Model.** Each production is described by a single smooth surface

$$Q(V,\varphi) = p_0(\varphi) + p_1(\varphi)\,V + \exp\big[p_2(\varphi) + p_3(\varphi)\,V\big],\qquad p_k(\varphi)=a_k+\exp(b_k\varphi+c_k)$$

fitted as **one global simultaneous fit** to all pooled points at once (`model.fit_global`). No floor is imposed: `p0` is free, so the charge decays to the no-gain bulk value on its own at high fluence. `config.P0_VS_PHI` toggles whether `p0` has its own fluence law (default) or is a single constant. The per-fluence fits `Q(V)=p0+p1·V+exp(p2+p3·V)` are kept as diagnostics. Charge error is `config.ERR_Q` (default 1 fC, switchable to 2).

Kernel: **Python (tb_analysis2)**. Full method: `docs/method.pdf`.

In [1]:
import ROOT
%jsroot on
import importlib
import config, data_io, model, plots
for m in (config, data_io, model, plots):
    importlib.reload(m)

# charge error: default 1 fC, switchable to 2
config.set_charge_error(1.0)

sensors = data_io.load_sensors()
print(data_io.summary(sensors))
print('\ncharge error = +-%.1f fC | param_mode = %s' % (config.ERR_Q, config.PARAM_MODE))

[HPK6]
   HPK6 S4-A                    setup=Beta phi=0.00e+00  n=10  V[100..190]  Q[7.55..40.32]
   HPK6 S2-B                    setup=Beta phi=0.00e+00  n=10  V[100..190]  Q[7.14..36.38]
   HPK2 W13-3 - 4e14            setup=Beta phi=4.00e+14  n= 7  V[150..375]  Q[3.00..29.10]
   HPK2 W4-25 - 8e14            setup=Beta phi=8.00e+14  n= 5  V[300..500]  Q[2.50..12.50]
   HPK6 S4-B - 1e15             setup=Beta phi=1.00e+15  n= 7  V[300..550]  Q[2.42..7.96]
   HPK6 S5-A - 1e15             setup=Beta phi=1.00e+15  n= 7  V[300..550]  Q[2.42..9.93]
   HPK6 S5-B - 1.5e15           setup=Beta phi=1.50e+15  n= 7  V[300..550]  Q[0.71..1.55]
   HPK6 S5-B - 1.5e15           setup=Beta phi=1.50e+15  n= 7  V[300..550]  Q[1.19..2.01]
[FBK-LF_PreSerie]
   FBK-LF W16-S13               setup=Beta phi=0.00e+00  n=10  V[120..210]  Q[8.00..30.60]
   FBK-LF W16-S52               setup=Beta phi=0.00e+00  n=10  V[120..210]  Q[7.90..31.01]
   FBK-LF W18-S2 - 1e15         setup=Beta phi=1.17e+15  n= 9  V[300.

In [2]:
# fit every production (sheet)
# (1) per-fluence Q(V) fits -> parameters p0..p3 (diagnostics, see tables below)
results = model.fit_all(sensors, verbose=True)

# (2) THE analytic model: ONE global simultaneous fit per production
#     Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V ),  p_k(phi)=a_k+exp(b_k*phi+c_k)
surfaces = model.fit_all_global(sensors)
print('\nAnalytic global law per production:')
for name, surf in surfaces.items():
    print('  ', name, '->', surf.law_string())


=== HPK6 (Beta) — 5 fluence points [param_mode=interp] ===
    HPK6_0.0e+00               n=20 chi2/ndf=  5.63/16 [root] p=['-1.694', '0.08425', '-4.936', '0.04275']
    HPK6_4.0e+14               n= 7 chi2/ndf=  0.18/3  [root] p=['2.203', '0.000282', '-2.61', '0.01575']
    HPK6_8.0e+14               n= 5 chi2/ndf=  0.06/1  [root] p=['-0.2589', '2.43e-05', '-1.344', '0.007795']
    HPK6_1.0e+15               n=14 chi2/ndf=  3.73/10 [root] p=['2.095', '9.906e-08', '-6.03', '0.01449']
    HPK6_1.5e+15               n=14 chi2/ndf=  0.74/10 [root] p=['0.8806', '1.505e-05', '-5.468', '0.009694']
  parametrization for HPK6:
    p0(phi) = interp[-1.69@0.0e+00, 2.2@4.0e+14, -0.259@8.0e+14, 2.1@1.0e+15, 0.881@1.5e+15]
    p1(phi) = interp[0.0842@0.0e+00, 0.000282@4.0e+14, 2.43e-05@8.0e+14, 9.91e-08@1.0e+15, 1.5e-05@1.5e+15]
    p2(phi) = interp[-4.94@0.0e+00, -2.61@4.0e+14, -1.34@8.0e+14, -6.03@1.0e+15, -5.47@1.5e+15]
    p3(phi) = interp[0.0428@0.0e+00, 0.0158@4.0e+14, 0.00779@8.0e+14, 0.014


=== FBK-LC1 (Beta) global simultaneous fit [p0 law + bulk anchor @ 3.0e+15, 5.2e+15, 9.0e+15, 1.5e+16] ===
  n = 38 points over 3 fluences   chi2/ndf = 81.9/27   RMS = 1.47 fC

Analytic global law per production:
   HPK6 -> Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.8405e-01 + exp(-6.8291e-14*phi +3.5528e-01)
       p1(phi) = +1.7232e-28 + exp(-2.2136e-15*phi -3.1882e+00)
       p2(phi) = -3.3470e-15*phi -3.1060e+00
       p3(phi) = +1.3935e-02 + exp(-2.8991e-15*phi -3.9129e+00)
       chi2/ndf = 25.4/49   RMS = 0.65 fC
   FBK-LF_PreSerie -> Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.8410e-01 + exp(-7.4565e-15*phi +1.9983e-01)
       p1(phi) = +7.7316e-38 + exp(-1.0104e-15*phi -4.3879e+00)
       p2(phi) = -1.2695e-15*phi -1.0002e+00
       p3(phi) = +8.4773e-03 + exp(-1.9741e-15*phi -4.4454e+00)
       chi2/ndf = 13.2/45   RMS = 0.49 fC
   FBK-LC1 -> Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
 

In [3]:
# Save every plot to Plots/ (overwrite) -- same set as `python run.py --save`:
# fit_QV / param_laws / param_pred / bias / charge_bias per production
# (+ the simulation overlay only where simulation/<prod>.csv exists), and
# LF-vs-LC1 style comparison plots for each family with >=2 productions.
outdir = plots.save_all(sensors, results, surfaces)
print('plots saved in', outdir)

plots saved in /Users/leonardolanteri/Desktop/Parametrization/Plots


## Q(V) fits — pooled data + fit (open red = test beam, cross-check only)

In [4]:
for name, r in results.items():
    plots.plot_QV(sensors, r).Draw()

## Parameter laws p₁(φ), p₂(φ), p₃(φ)
The three fluence-dependent parameters of the global surface, each with its exponential law (blue) and the uncertainty band propagated from the global fit. The markers lie on the laws by construction — it is one simultaneous fit.

In [5]:
for name, surf in surfaces.items():
    plots.plot_param_laws(surf).Draw()

## The analytic model vs data
The single global surface `Q(V,φ) = p0 + p1(φ)·V + exp(p2(φ) + p3(φ)·V)` overlaid on the pooled data at each measured fluence — one formula describing every fluence.

In [6]:
for name, surf in surfaces.items():
    plots.plot_surface_vs_data(sensors, surf).Draw()

## Bias to reach the target charge, vs fluence

In [7]:
for name, surf in surfaces.items():            # one figure per sheet
    plots.plot_bias_for_target({name: surf}, Q_target=config.Q_TARGET, phi_hi=1.6e15).Draw()
print('Bias to reach Q = %.0f fC:' % config.Q_TARGET)
for name, surf in surfaces.items():
    row = {}
    for phi in surf.fluences:
        v = surf.V_for_Q(config.Q_TARGET, phi)
        row['%.1e' % phi] = 'n/a' if v != v else '%.0f V' % v
    print(' ', name, row)

Bias to reach Q = 8 fC:
  HPK6 {'0.0e+00': '107 V', '4.0e+14': '272 V', '8.0e+14': '456 V', '1.0e+15': '537 V', '1.5e+15': '704 V'}
  FBK-LF_PreSerie {'0.0e+00': '126 V', '1.2e+15': '441 V', '1.8e+15': '573 V'}
  FBK-LC1 {'0.0e+00': '104 V', '1.0e+15': '381 V', '1.5e+15': '590 V'}


## The analytic surface Q(V,φ) as a 2-D map

In [8]:
tf2s = {name: surf.tf2() for name, surf in surfaces.items()}
c = ROOT.TCanvas('csurf', 'surfaces', 500 * len(tf2s), 450)
c.Divide(len(tf2s), 1)
for i, (name, f2) in enumerate(tf2s.items()):
    c.cd(i + 1); ROOT.gPad.SetRightMargin(0.16)
    f2.SetMaximum(20); f2.Draw('colz')
c.Draw()

## Operating point: charge obtainable & bias needed vs fluence (SEB-capped)

**Cell 1** — for each production: dashed = bias needed to reach the target charge (15 fC), capped at the SEB limit `V_SEB` (HPK 550 V, FBK 600 V); solid = charge actually obtainable = `min(target, Q(V_SEB, φ))` (flat at the target until the SEB knee, then the charge available at `V_SEB`).

**Cell 2** — fix a charge (e.g. 8 fC) and read off the bias needed at each fluence (flagged if it exceeds `V_SEB`).

In [9]:
# === CELL 1 — charge obtained & bias needed vs fluence, with the SEB cap ===
# Uses the global surface (computed above): one smooth trend, charge converges on
# its own to the free bulk term p0 (~0.3-0.5 fC) at high fluence (no floor imposed).
TARGET = config.PLATEAU_CHARGE            # 15 fC (solid-line plateau)
# V_SEB per production from config.V_SEB (HPK 550 V, FBK 600 V); override with vmax={...}
# one separate figure per sheet; add overlay='auto' to superimpose simulation/<prod>.csv
for name, surf in surfaces.items():
    plots.plot_charge_bias_vs_fluence({name: surf}, target=TARGET, vmax=None).Draw()

In [10]:
# === CELL 2 — fix a charge, get the bias needed at each fluence ===
Q_FIX = 8.0   # fC  (change me)

for name, surf in surfaces.items():            # one figure per sheet
    plots.plot_bias_for_target({name: surf}, Q_target=Q_FIX, phi_hi=1.6e15).Draw()

print('Bias needed to reach Q = %.1f fC (global surface):' % Q_FIX)
for name, surf in surfaces.items():
    vmax = plots._vmax_for(name, None)
    for phi in surf.fluences:
        v = surf.V_for_Q(Q_FIX, phi, v_hi=1200)
        if v != v:
            s = 'not reachable'
        else:
            s = '%.0f V' % v + ('  (> V_SEB = %.0f V!)' % vmax if v > vmax else '')
        print('  %-6s  phi=%.2e  ->  %s' % (name, phi, s))

Bias needed to reach Q = 8.0 fC (global surface):
  HPK6    phi=0.00e+00  ->  107 V
  HPK6    phi=4.00e+14  ->  272 V
  HPK6    phi=8.00e+14  ->  456 V
  HPK6    phi=1.00e+15  ->  537 V
  HPK6    phi=1.50e+15  ->  704 V  (> V_SEB = 550 V!)
  FBK-LF_PreSerie  phi=0.00e+00  ->  126 V
  FBK-LF_PreSerie  phi=1.17e+15  ->  441 V
  FBK-LF_PreSerie  phi=1.76e+15  ->  573 V
  FBK-LC1  phi=0.00e+00  ->  104 V
  FBK-LC1  phi=1.00e+15  ->  381 V
  FBK-LC1  phi=1.50e+15  ->  590 V


## Parameters of the production
Per-fluence Q(V) fit parameters `p0..p3` (with errors and χ²/ndf) and the vs-fluence laws actually used. Also written to `parameters_<prod>.csv`.

In [11]:
# === Parameters of each production ===
# The global surface  Q(V,phi) = p0 + p1(phi)*V + exp(p2(phi) + p3(phi)*V):
#   p0,p1,p2,p3  values of the fitted laws at each measured fluence (+/- band)
#   (p0 is a law when config.P0_VS_PHI, else a single constant)
import pandas as pd
for name, surf in surfaces.items():
    print('=' * 72)
    print(name, '   (p0 %s)' % ('law(phi)' if not surf.p0_const else 'const'))
    df = pd.DataFrame({
        'fluence': surf.fluences,
        'p0': surf.p0_nodes, 'p0_err': surf.p0_err,
        'p1': surf.p1_nodes, 'p1_err': surf.p1_err,
        'p2': surf.p2_nodes, 'p2_err': surf.p2_err,
        'p3': surf.p3_nodes, 'p3_err': surf.p3_err,
    })
    display(df.round(5))
    print('analytic laws:')
    print(surf.law_string())
    df.to_csv('parameters_%s.csv' % model._short(name), index=False)
    print('-> saved parameters_%s.csv' % model._short(name))

HPK6    (p0 law(phi))


,fluence,p0,p0_err,p1,p1_err,p2,p2_err,p3,p3_err
0,0.000000e+00,1.91063,1.29313,0.04125,0.01539,-3.10597,0.73137,0.03392,0.00358
1,4.000000e+14,0.48405,0.23651,0.01702,0.00451,-4.44479,0.88533,0.02020,0.00223
2,8.000000e+14,0.48405,0.23651,0.00702,0.00190,-5.78361,1.37973,0.01590,0.00269
3,1.000000e+15,0.48405,0.23651,0.00451,0.00147,-6.45302,1.67038,0.01504,0.00288
4,1.500000e+15,0.48405,0.23651,0.00149,0.00092,-8.12655,2.44111,0.01419,0.00313


analytic laws:
Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.8405e-01 + exp(-6.8291e-14*phi +3.5528e-01)
       p1(phi) = +1.7232e-28 + exp(-2.2136e-15*phi -3.1882e+00)
       p2(phi) = -3.3470e-15*phi -3.1060e+00
       p3(phi) = +1.3935e-02 + exp(-2.8991e-15*phi -3.9129e+00)
       chi2/ndf = 25.4/49   RMS = 0.65 fC
-> saved parameters_HPK6.csv
FBK-LF_PreSerie    (p0 law(phi))


,fluence,p0,p0_err,p1,p1_err,p2,p2_err,p3,p3_err
0,0.000000e+00,1.70530,2.53780,0.01243,0.03840,-1.00024,1.16905,0.02021,0.00453
1,1.170000e+15,0.48430,1.24867,0.00381,0.00498,-2.48561,0.70153,0.00964,0.00107
2,1.755000e+15,0.48411,0.25823,0.00211,0.00201,-3.22829,0.83142,0.00884,0.00123


analytic laws:
Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.8410e-01 + exp(-7.4565e-15*phi +1.9983e-01)
       p1(phi) = +7.7316e-38 + exp(-1.0104e-15*phi -4.3879e+00)
       p2(phi) = -1.2695e-15*phi -1.0002e+00
       p3(phi) = +8.4773e-03 + exp(-1.9741e-15*phi -4.4454e+00)
       chi2/ndf = 13.2/45   RMS = 0.49 fC
-> saved parameters_FBKLF_PreSerie.csv
FBK-LC1    (p0 law(phi))


,fluence,p0,p0_err,p1,p1_err,p2,p2_err,p3,p3_err
0,0.000000e+00,3.42037,7.38050,0.01888,0.08808,-1.50558,1.81755,0.02370,0.00702
1,1.000000e+15,1.10893,3.07227,0.00418,0.01568,-2.28237,1.59604,0.01035,0.00266
2,1.500000e+15,0.77418,1.83793,0.00197,0.00785,-2.67076,1.90163,0.00758,0.00244


analytic laws:
Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.8926e-01 + exp(-1.5539e-15*phi +1.0754e+00)
       p1(phi) = +1.0691e-39 + exp(-1.5074e-15*phi -3.9695e+00)
       p2(phi) = -7.7679e-16*phi -1.5056e+00
       p3(phi) = +3.9048e-03 + exp(-1.1217e-15*phi -3.9223e+00)
       chi2/ndf = 81.9/27   RMS = 1.47 fC
-> saved parameters_FBKLC1.csv
